## 🎯 Learning Objectives
* Load and prepare a sentiment analysis dataset using the Hugging Face `datasets` library.
* Utilize a pre-trained BERT model and tokenizer from Hugging Face `transformers` for sequence classification.
* Implement a fine-tuning pipeline for a BERT model using the Hugging Face `Trainer` API.
* Evaluate the performance of the fine-tuned model using appropriate metrics like accuracy and F1-score.
* Understand the practical steps involved in adapting large language models for downstream tasks.


## DL02-L13: Exercise - Fine-tune a BERT model for sentiment analysis

### Task Description

In this exercise, you will fine-tune a pre-trained BERT model for a sentiment analysis task. Sentiment analysis is a classic NLP problem where the goal is to classify the sentiment of a given text (e.g., positive, negative, neutral).

We will use a subset of the **SST-2 (Stanford Sentiment Treebank v2)** dataset, which is a binary classification task (positive/negative sentiment). The Hugging Face `datasets` library provides easy access to this dataset.

Your primary goal is to leverage the Hugging Face `transformers` and `datasets` libraries to efficiently fine-tune a `bert-base-uncased` model.

### Requirements

1.  **Load Dataset**: Load the `sst2` dataset from the Hugging Face `datasets` library. You should focus on the `train` and `validation` splits.
2.  **Load Tokenizer & Model**: Load the `bert-base-uncased` tokenizer and `AutoModelForSequenceClassification` from `transformers`.
3.  **Preprocess Data**: Tokenize the dataset using the loaded tokenizer. Ensure the tokenization function is applied to both training and validation sets. Use `DataCollatorWithPadding` for dynamic padding.
4.  **Define Training Arguments**: Set up `TrainingArguments` for the `Trainer`. Consider batch size, learning rate, number of epochs, and evaluation strategy.
5.  **Define Metrics**: Create a `compute_metrics` function that calculates accuracy and F1-score for the classification task.
6.  **Initialize and Train `Trainer`**: Instantiate the `Trainer` with your model, arguments, datasets, tokenizer, and `compute_metrics` function. Then, start the training process.
7.  **Evaluate Model**: After training, evaluate the model on the validation set and print the results.

### Evaluation Criteria

*   **Correctness**: The code runs without errors and successfully fine-tunes the BERT model.
*   **Performance**: The fine-tuned model achieves reasonable accuracy and F1-score on the validation set (e.g., >85% accuracy is a good target for SST-2).
*   **Clarity & Readability**: The code is well-structured, easy to understand, and includes comments where necessary.
*   **Hugging Face Best Practices**: Proper use of `transformers` `Trainer` API, `datasets` library, and `evaluate` library for metrics.
*   **Efficiency**: Efficient data loading and processing (e.g., using `map` for tokenization, `DataCollatorWithPadding`).


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install transformers datasets accelerate evaluate torch -q

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset, DatasetDict
import evaluate
import numpy as np

# --- Configuration --- 
MODEL_CHECKPOINT = "bert-base-uncased"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
OUTPUT_DIR = "./results_sentiment_bert"

# --- 1. Load Dataset --- 
print(f"Loading dataset 'sst2'...")
# We'll load a small subset for quicker iteration during development/exercise
# In a real scenario, you'd use the full dataset.
raw_datasets = load_dataset("sst2")

# For demonstration, let's take a smaller subset if the full dataset is too large
# This is optional, remove if you want to train on the full dataset
# raw_datasets['train'] = raw_datasets['train'].select(range(1000))
# raw_datasets['validation'] = raw_datasets['validation'].select(range(200))

print(f"Dataset loaded: {raw_datasets}")

# --- 2. Load Tokenizer --- 
print(f"Loading tokenizer '{MODEL_CHECKPOINT}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# --- 3. Preprocess Data --- 
def tokenize_function(examples):
    # The 'sentence' column contains the text for sentiment analysis
    return tokenizer(examples["sentence"], truncation=True, padding=False)

print("Tokenizing datasets...")
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Remove original text column and rename 'label' to 'labels' for Trainer compatibility
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Set format to PyTorch tensors
tokenized_datasets.set_format("torch")

# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# --- 4. Define Metrics --- 
# Load the 'glue' metric, specifically for 'sst2'
metric = evaluate.load("glue", "sst2")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Setup complete. Ready for student implementation.")


### Your Implementation

Now it's your turn! Using the pre-loaded `tokenizer`, `tokenized_datasets`, `data_collator`, and `compute_metrics` function, complete the following steps:

1.  **Load the `AutoModelForSequenceClassification`** for `bert-base-uncased`. Remember that SST-2 is a binary classification task (2 labels).
2.  **Define `TrainingArguments`**: Configure the training process. Pay attention to `output_dir`, `learning_rate`, `per_device_train_batch_size`, `per_device_eval_batch_size`, `num_train_epochs`, `evaluation_strategy`, `save_strategy`, and `load_best_model_at_end`.
3.  **Initialize the `Trainer`**: Pass the model, training arguments, train dataset, eval dataset, tokenizer, data collator, and `compute_metrics` function to the `Trainer` constructor.
4.  **Train the model**: Call the `train()` method on your `Trainer` instance.
5.  **Evaluate the model**: Call the `evaluate()` method on your `Trainer` instance and print the results.

Good luck!


In [ ]:
# --- Solution --- 

# 1. Load the AutoModelForSequenceClassification
# SST-2 has 2 labels (positive/negative)
print(f"Loading model '{MODEL_CHECKPOINT}' for sequence classification with 2 labels...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

# 2. Define TrainingArguments
print("Defining TrainingArguments...")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,                                # Directory to save checkpoints and logs
    learning_rate=LEARNING_RATE,                          # Learning rate for the optimizer
    per_device_train_batch_size=BATCH_SIZE,               # Batch size per GPU/CPU for training
    per_device_eval_batch_size=BATCH_SIZE,                # Batch size per GPU/CPU for evaluation
    num_train_epochs=NUM_EPOCHS,                          # Total number of training epochs
    weight_decay=0.01,                                    # Apply weight decay to avoid overfitting
    evaluation_strategy="epoch",                          # Evaluate at the end of each epoch
    save_strategy="epoch",                                # Save model checkpoint at the end of each epoch
    load_best_model_at_end=True,                          # Load the best model found during training at the end
    metric_for_best_model="accuracy",                     # Metric to use for determining the best model
    push_to_hub=False,                                    # Set to True if you want to push to Hugging Face Hub
    report_to="none"                                      # Disable reporting to external services like wandb
)

# 3. Initialize the Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,                                          # The model to train
    args=training_args,                                   # Training arguments
    train_dataset=tokenized_datasets["train"],            # Training dataset
    eval_dataset=tokenized_datasets["validation"],        # Evaluation dataset
    tokenizer=tokenizer,                                  # Tokenizer for data collation
    data_collator=data_collator,                          # Data collator for dynamic padding
    compute_metrics=compute_metrics                       # Function to compute metrics during evaluation
)

# 4. Train the model
print("Starting model training...")
trainer.train()

# 5. Evaluate the model
print("Evaluating the fine-tuned model on the validation set...")
eval_results = trainer.evaluate()

print("\n--- Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

print("\nFine-tuning and evaluation complete!")

# Optional: Save the fine-tuned model and tokenizer
# trainer.save_model("./fine_tuned_bert_sst2")
# tokenizer.save_pretrained("./fine_tuned_bert_sst2")
# print("Model and tokenizer saved to './fine_tuned_bert_sst2'")
